# Zernike响应矩阵校准 - 硬件调试notebook

本notebook用于实际硬件调试，执行Zernike响应矩阵校准。

## 硬件连接
- **SLM**: Santec SLM-200 (通过ZernikeSLM封装)
- **WFS**: Thorlabs WFS

## 校准原理

使用正负扰动测量消除系统偏置:

$R = \frac{R^+ - R^-}{2\cdot\delta}$

其中 $R^+$, $R^-$ 是正负扰动下的WFS响应，$\delta$ 是扰动幅度(波长单位)。

In [ ]:
# %% [markdown]
# ## 1. 导入核心模块
import sys
sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time
from datetime import datetime

# Zernike响应矩阵模块
from ao_shaping.optimizer.wf.zernike_response_matrix import (
    calibrate_zernike_response_matrix,
    measure_zernike_mode_response,
    ZernikeResponseMatrixResult,
    set_slm_flat,
    save_zernike_response_matrix,
    load_zernike_response_matrix,
    DEFAULT_N_MAX,
    DEFAULT_MAGNITUDE,
    DEFAULT_N_AVERAGES,
    DEFAULT_N_CYCLES,
    DEFAULT_WAIT_TIME,
)

# 工具函数
from ao_shaping.utils.matrix_utils import (
    calc_n_zernike_terms,
    compute_pinv,
    compute_lstsq,
)

print("✓ 核心模块导入成功")
print(f"  - 默认n_max: {DEFAULT_N_MAX}")
print(f"  - 默认扰动幅度: {DEFAULT_MAGNITUDE} λ")
print(f"  - 默认采样次数: {DEFAULT_N_AVERAGES}")
print(f"  - 默认循环次数: {DEFAULT_N_CYCLES}")

In [ ]:
# %% [markdown]
# ## 2. 校准参数配置

# ==================== 核心参数 (可根据需要修改) ====================
N_MAX = 10                      # Zernike最大阶数
MAGNITUDE = 0.5                 # 扰动幅度 (波长单位)
N_CYCLES = 3                    # 正负交替循环次数
N_AVERAGES = 20                 # 每次WFS读取取平均次数
WAIT_TIME = 0.1                 # 施加相位后等待时间 (秒)
EXCLUDED_PISTON = True          # 是否排除piston (Z1)
EXCLUDED_TIP_TILT = True        # 是否排除tip/tilt (Z2, Z3)

# 输出路径
OUTPUT_DIR = Path("data/zernike_response_matrix")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 计算项数
n_remove = (1 if EXCLUDED_PISTON else 0) + (2 if EXCLUDED_TIP_TILT else 0)
n_slm_terms = calc_n_zernike_terms(N_MAX) - n_remove

print("="*60)
print("校准参数配置")
print("="*60)
print(f"  Zernike最大阶数 (n_max): {N_MAX}")
print(f"  扰动幅度: {MAGNITUDE} λ")
print(f"  正负循环次数: {N_CYCLES}")
print(f"  平均采样次数: {N_AVERAGES}")
print(f"  等待时间: {WAIT_TIME} s")
print(f"  排除piston: {EXCLUDED_PISTON}")
print(f"  排除tip/tilt: {EXCLUDED_TIP_TILT}")
print("-"*60)
print(f"  SLM项数: {n_slm_terms}")
print(f"  输出目录: {OUTPUT_DIR}")
print("="*60)

In [ ]:
# %% [markdown]
# ## 3. 初始化硬件设备

from ao_shaping.drivers.slm.zernike_slm import ZernikeSLM
from ao_shaping.drivers.wfs.thorlab_wfs import WFSManager, MlaRes

# SLM配置
SLM_NUMBER = 1
SLM_WAVELENGTH = 1064  # nm
SLM_N_MAX = N_MAX

# WFS配置 (可选, 如不设置则使用配置文件)
WFS_RESOLUTION = MlaRes.Res768  # 可选: Res320, Res512, Res768, Res1024, Res1280
WFS_EXPOSURE_TIME = 10.0  # ms, 0=自动

print("初始化设备...")
print(f"  SLM: Santec SLM-{SLM_NUMBER}, λ={SLM_WAVELENGTH}nm, n_max={SLM_N_MAX}")
print(f"  WFS: 分辨率={WFS_RESOLUTION.name}, 曝光={WFS_EXPOSURE_TIME}ms")

# 初始化设备 (使用context manager)
zslm = ZernikeSLM(
    slm_number=SLM_NUMBER,
    wavelength=SLM_WAVELENGTH,
    n_max=SLM_N_MAX,
)
wfs = WFSManager(
    mla_index=WFS_RESOLUTION,
    exp_time=WFS_EXPOSURE_TIME,
)

try:
    # 打开设备
    zslm.open()
    print("✓ SLM已打开")
    
    wfs.initialize()
    print("✓ WFS已初始化")
    
    # 获取设备信息
    slm_info = zslm.get_hardware_info()
    print(f"  SLM信息: {slm_info}")
    print(f"  WFS序列号: {wfs.serial_num}")
    print(f"  WFS分辨率: {wfs.mla_index.name}, 子孔径数: {wfs.num_spots_x}x{wfs.num_spots_y}")
    
    DEVICE_READY = True
    
except Exception as e:
    print(f"✗ 设备初始化失败: {e}")
    DEVICE_READY = False

In [ ]:
# %% [markdown]
# ## 4. 设置SLM平相位 + 预热WFS

if not DEVICE_READY:
    raise RuntimeError("设备未就绪，请检查硬件连接")

# 设置SLM为平相位
set_slm_flat(zslm._slm)
print("✓ SLM已设置为平相位")

# 预热WFS - 采集多帧建立稳定读数
print("预热WFS (采集10帧)...")
for i in range(10):
    wfs.take_image()
    time.sleep(0.05)
    if i == 0:
        # 读取初始Zernike系数
        init_coeffs = wfs.get_zernike(zernike_order=N_MAX)
        print(f"  初始Zernike系数 (前5项): {init_coeffs[:5]}")

print("✓ WFS预热完成")

In [ ]:
# %% [markdown]
# ## 5. 快速单模式测试 (在完整校准前验证)

# 测试单个Zernike模式 (Z4 = focus) 验证流程正确性
TEST_MODE_INDEX = 1  # 对应Z4 (focus), 排除piston和tip-tilt后
TEST_MAGNITUDE = 0.5  # λ

print("="*60)
print(f"单模式测试: mode_index={TEST_MODE_INDEX}, magnitude={TEST_MAGNITUDE}λ")
print("="*60)

# 构建测试系数
n_full = wfs.calc_n_zernike_terms(N_MAX)
test_coeffs = np.zeros(n_full, dtype=np.float64)

# excluded_piston=True, excluded_tip_tilt=True => noll_offset=3
noll_offset = 3
test_coeffs[TEST_MODE_INDEX + noll_offset] = TEST_MAGNITUDE

print(f"测试系数: Z{TEST_MODE_INDEX + noll_offset + 1} = {TEST_MAGNITUDE}λ")

# 测量响应
mean_resp, var_resp, mean_dev, var_dev = measure_zernike_mode_response(
    zslm=zslm,
    wfs=wfs,
    coefficients=test_coeffs,
    coeff_value=TEST_MAGNITUDE,
    n_averages=10,  # 快速测试用较少采样
    n_cycles=1,     # 单次循环
    wait_time=WAIT_TIME,
    excluded_piston=EXCLUDED_PISTON,
    excluded_tip_tilt=EXCLUDED_TIP_TILT,
    zernike_order=N_MAX,
    mode_index=TEST_MODE_INDEX,
)

# 恢复平相位
set_slm_flat(zslm._slm)

print(f"\n测量结果:")
print(f"  响应向量长度: {len(mean_resp)}")
print(f"  响应向量前10项: {mean_resp[:10]}")
print(f"  响应RMS: {np.sqrt(np.mean(mean_resp**2)):.6f}")
print(f"  平均方差: {np.mean(var_resp):.6f}")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax1 = axes[0]
ax1.bar(range(len(mean_resp)), mean_resp, color='steelblue', alpha=0.8)
ax1.set_xlabel('WFS Zernike Index')
ax1.set_ylabel('Response')
ax1.set_title(f'单模式响应 (Z{TEST_MODE_INDEX + noll_offset + 1})')
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.bar(range(len(var_resp)), var_resp, color='coral', alpha=0.8)
ax2.set_xlabel('WFS Zernike Index')
ax2.set_ylabel('Variance')
ax2.set_title('响应方差')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ 单模式测试完成，如结果正常可继续执行完整校准")

In [ ]:
# %% [markdown]
# ## 6. 执行完整响应矩阵校准

# 恢复平相位后开始校准
set_slm_flat(zslm._slm)
time.sleep(0.5)

print("="*60)
print("开始Zernike响应矩阵校准")
print("="*60)
print(f"参数: n_max={N_MAX}, magnitude={MAGNITUDE}λ, cycles={N_CYCLES}, averages={N_AVERAGES}")
print(f"预计耗时: ~{n_slm_terms * N_CYCLES * N_AVERAGES * 0.5 / 60:.1f} 分钟")
print("="*60)

start_time = time.time()

result = calibrate_zernike_response_matrix(
    zslm=zslm,
    wfs=wfs,
    n_max=N_MAX,
    magnitude=MAGNITUDE,
    n_cycles=N_CYCLES,
    n_averages=N_AVERAGES,
    wait_time=WAIT_TIME,
    excluded_piston=EXCLUDED_PISTON,
    excluded_tip_tilt=EXCLUDED_TIP_TILT,
    compute_inverses=True,
    verbose=True,
)

elapsed_time = time.time() - start_time

print("="*60)
print(f"校准完成! 耗时: {elapsed_time/60:.1f} 分钟")
print("="*60)
print(f"响应矩阵形状: {result.matrix.shape}")
print(f"WFS项数: {result.n_wfs_terms}")
print(f"SLM项数: {result.n_slm_terms}")
print(f"平均方差: {result.mean_variance:.6f}")
print(f"条件数: {result.condition_number:.2e}" if result.condition_number else "N/A")

In [ ]:
# %% [markdown]
# ## 7. 保存校准结果

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_path = OUTPUT_DIR / f"zernike_response_matrix_{timestamp}.h5"

save_zernike_response_matrix(result, str(save_path), include_inverses=True)
print(f"✓ 响应矩阵已保存: {save_path}")

# 同时保存JSON元数据
json_path = OUTPUT_DIR / f"zernike_response_matrix_{timestamp}.json"
import json
metadata = {
    "n_max": result.n_max,
    "magnitude": result.magnitude,
    "wavelength_nm": result.wavelength_nm,
    "n_averages": result.n_averages,
    "n_cycles": result.n_cycles,
    "timestamp": result.timestamp,
    "excluded_piston": result.excluded_piston,
    "excluded_tip_tilt": result.excluded_tip_tilt,
    "mean_variance": result.mean_variance,
    "condition_number": result.condition_number,
}
with open(json_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ 元数据已保存: {json_path}")

In [ ]:
# %% [markdown]
# ## 8. 可视化分析

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. 响应矩阵热图
ax1 = axes[0, 0]
im1 = ax1.imshow(result.matrix, aspect='auto', cmap='RdBu_r',
                 vmin=-np.max(np.abs(result.matrix))*0.5, vmax=np.max(np.abs(result.matrix))*0.5)
ax1.set_xlabel('SLM Mode Index')
ax1.set_ylabel('WFS Mode Index')
ax1.set_title(f'响应矩阵 (n_max={result.n_max})')
fig.colorbar(im1, ax=ax1, label='Response')

# 2. 方差矩阵热图
ax2 = axes[0, 1]
im2 = ax2.imshow(result.variance_matrix, aspect='auto', cmap='YlOrRd')
ax2.set_xlabel('SLM Mode Index')
ax2.set_ylabel('WFS Mode Index')
ax2.set_title(f'方差矩阵 (mean={result.mean_variance:.6f})')
fig.colorbar(im2, ax=ax2, label='Variance')

# 3. 每列平均方差
ax3 = axes[1, 0]
col_mean_var = np.mean(result.variance_matrix, axis=0)
ax3.bar(range(len(col_mean_var)), col_mean_var, color='steelblue', alpha=0.8)
ax3.set_xlabel('SLM Mode Index')
ax3.set_ylabel('Mean Variance')
ax3.set_title('各模式测量稳定性')
ax3.grid(True, alpha=0.3)

# 4. SVD奇异值
ax4 = axes[1, 1]
if result.pinv_matrix is not None:
    U, s, Vt = np.linalg.svd(result.matrix)
    ax4.plot(s, 'o-', markersize=4, color='darkorange')
    ax4.set_xlabel('Singular Value Index')
ax4.set_ylabel('Singular Value')
ax4.set_title(f'SVD (condition={result.condition_number:.2e})')
ax4.set_yscale('log')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ 可视化完成")

In [ ]:
# %% [markdown]
# ## 9. 验证逆矩阵 (测试校正功能)

if result.pinv_matrix is None:
    print("跳过验证: 未计算伪逆矩阵")
else:
    # 读取当前波前作为测试
    wfs.take_image()
    test_wavefront = wfs.get_zernike(zernike_order=N_MAX)
    
    # 排除piston和tip-tilt后的有效部分
    start_idx = 3 if (EXCLUDED_PISTON and EXCLUDED_TIP_TILT) else (1 if EXCLUDED_PISTON else 0)
    test_vector = test_wavefront[start_idx:start_idx+n_slm_terms]
    
    # 使用伪逆计算校正指令
    correction = result.pinv_matrix @ test_vector
    
    # 验证效果
    residual = result.matrix @ correction - test_vector
    rms_before = np.sqrt(np.mean(test_vector**2))
    rms_after = np.sqrt(np.mean(residual**2))
    
    print("="*60)
    print("逆矩阵验证")
    print("="*60)
    print(f"校正前RMS: {rms_before:.6f} λ")
    print(f"校正后RMS: {rms_after:.6f} λ")
    print(f"RMS改善: {(rms_before - rms_after) / rms_before * 100:.1f}%")
    
    # 可视化
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].bar(range(len(test_vector)), test_vector, alpha=0.7, color='coral')
    axes[0].set_title('原始波前')
    axes[0].set_xlabel('Mode Index')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].bar(range(len(correction)), correction, alpha=0.7, color='steelblue')
    axes[1].set_title('校正指令')
    axes[1].set_xlabel('Mode Index')
    axes[1].grid(True, alpha=0.3)
    
    axes[2].bar(range(len(residual)), residual, alpha=0.7, color='forestgreen')
    axes[2].set_title(f'残余 (RMS={rms_after:.4f})')
    axes[2].set_xlabel('Mode Index')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 恢复平相位
    set_slm_flat(zslm._slm)

In [ ]:
# %% [markdown]
# ## 10. 关闭设备

# 清理资源
set_slm_flat(zslm._slm)
print("✓ 已恢复SLM平相位")

# 关闭WFS
wfs.close()
print("✓ WFS已关闭")

# 关闭SLM
zslm.close()
print("✓ SLM已关闭")

print("\n" + "="*60)
print("校准流程完成!")
print("="*60)
print(f"输出文件:")
print(f"  - 响应矩阵: {save_path}")
print(f"  - 元数据: {json_path}")
print("="*60)

In [ ]:
# %% [markdown]
# ## 11. 后续使用 - 加载已保存的响应矩阵

# # 加载之前保存的响应矩阵
# loaded_result = load_zernike_response_matrix("path/to/your/matrix.h5")
# print(f"Loaded: {loaded_result.matrix.shape}")
# 
# # 使用进行波前校正
# # 1. 读取当前波前
# wfs.take_image()
# current_coeffs = wfs.get_zernike(zernike_order=10)
# 
# # 2. 计算校正指令
# valid_coeffs = current_coeffs[3:]  # 排除piston, tip, tilt
# correction = loaded_result.pinv_matrix @ valid_coeffs
# 
# # 3. 发送到SLM
# zslm.send_zernike(correction)